# Module 9: Budget check (job task 2)
Finds categories that are **over budget in the latest month**, and passes the count
to the next task in the job using **task values**.

Parameters come in through **widgets**, so the job (not the code) decides which schema to check.

In [0]:
# Widgets = notebook parameters. When the job runs this notebook, it can override the defaults.
dbutils.widgets.text("schema", "finance_pipeline", "Schema with the gold tables")

schema = dbutils.widgets.get("schema")
print(f"Checking budgets in workspace.{schema}")

In [0]:
over_budget = spark.sql(f"""
    SELECT month, category, monthly_budget, spent, round(pct_used * 100, 1) AS pct_used
    FROM workspace.{schema}.gold_budget_status
    WHERE status = 'OVER'
      AND month = (SELECT max(month) FROM workspace.{schema}.gold_budget_status)
    ORDER BY pct_used DESC
""")

display(over_budget)
over_count = over_budget.count()
print(f"Categories over budget in the latest month: {over_count}")

In [0]:
# Hand the result to the next task. The If/else task reads it as
#   {{tasks.budget_check.values.over_count}}
# (Outside a job this line just prints a note; that's normal.)
dbutils.jobs.taskValues.set(key="over_count", value=over_count)